### Makemore Bigram / Trigram Exercises: NLL and Cross-Entropy Super Edition

> **A good probabilistic model should not be very surprised when it sees a real sample.**  
> NLL measures exactly that surprise. Cross-entropy is the expected version of the same idea between a true distribution and a model distribution.

You will see the same core object repeatedly:

$$
L(\theta) = -\frac{1}{N}\sum_{i=1}^N \log p_\theta(y_i\mid x_i)
$$

In the bigram / trigram name model:

- $x_i$: the context, such as the current character `e`, or the two previous characters `em`
- $y_i$: the true next character, such as `m`
- $y$: any possible next character, `a,b,c,...,z,.`
- $p_\theta(y\mid x_i)$: the model's softmax distribution over the next character
- $-\log p_\theta(y_i\mid x_i)$: the surprise / NLL for this one real sample

1. Count tables and neural bigram/trigram models estimate the same conditional distribution.
2. NLL, cross-entropy, and `F.cross_entropy` are the same idea at different levels of abstraction.
3. Train/dev/test, smoothing, and temperature each belong to different parts of the ML workflow.


#### 0. Setup: imports, data, vocabulary

1. Load `names.txt`.
2. Build the character vocabulary: `.` is the start/end token, and the other characters are `a-z`.
3. Assign an integer id to every character so the model can process them.

In [ ]:
from pathlib import Path
import math
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

torch.set_printoptions(precision=4, sci_mode=False)

g = torch.Generator().manual_seed(2147483647)

names_path = Path('names.txt')
if not names_path.exists():
    url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
    try:
        print('names.txt not found locally; trying to download...')
        urllib.request.urlretrieve(url, names_path)
    except Exception as e:
        raise FileNotFoundError(
            "names.txt was not found and automatic download failed. "
            "Please place names.txt in the same directory as this notebook."
        ) from e

words = names_path.read_text().splitlines()
print('num words:', len(words))
print('first 10 words:', words[:10])


In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)

print('stoi:', stoi)
print('itos:', itos)
print('vocab_size:', vocab_size)


#### 1. Notation alignment: from abstract ML to makemore

$$
(x_i, y_i), \quad y, \quad p(y\mid x_i), \quad p_\theta(y\mid x_i), \quad -\log p_\theta(y_i\mid x_i)
$$

maps directly onto makemore.

For the name:

```text
.emma.
```

The bigram samples are:

```text
. -> e
e -> m
m -> m
m -> a
a -> .
```

For the sample `e -> m`:

- $x_i = e$: the current character
- $y_i = m$: the true next character
- $y$: any possible character, such as `a,b,c,...,z,.`
- $p_\theta(y\mid e)$: the full 27-class probability distribution produced by the model
- $-\log p_\theta(m\mid e)$: the NLL for this sample

For a trigram model:

```text
.. -> e
.e -> m
em -> m
mm -> a
ma -> .
```

Now $x_i$ is a two-character context, such as `em`, while $y_i$ is still the next character.


In [ ]:
def show_examples(word='emma', block_size=1):
    context = [0] * block_size
    print(f'word with end token: {word}.')
    for ch in word + '.':
        ix = stoi[ch]
        ctx = ''.join(itos[i] for i in context)
        print(f'{ctx:>{block_size}} -> {ch}')
        context = context[1:] + [ix]

print('Bigram examples:')
show_examples('emma', block_size=1)
print('\nTrigram examples:')
show_examples('emma', block_size=2)


#### 2. Helpers: datasets, counts, NLL

These helpers turn names into `context -> next character` examples, estimate count-based probabilities, and evaluate NLL.

##### `build_dataset(words, block_size)`

Creates $X=\text{context}$ and $Y=\text{next character}$.

- `block_size=1`: bigram, one previous character.
- `block_size=2`: trigram, two previous characters.

##### `count_ngrams(X,Y,block_size)`

Builds a count table. We write counts as $N(x,y)$ to match the code variable `N`:

$$
N(x,y) = \#\{\text{times context }x\text{ is followed by }y\}
$$

##### `probs_from_counts(N, smoothing)`

Normalizes counts into conditional probabilities with add-$s$ smoothing:

$$
\hat p_s(y\mid x) = \frac{N(x,y)+s}{N(x)+s|\mathcal{Y}|}, \qquad N(x)=\sum_{y'}N(x,y')
$$

##### `nll_from_examples(P,X,Y)` and `nll_from_counts(N,P)`

These compute the same average surprise, either example by example or count-weighted:

$$
-\frac{1}{M}\sum_i \log P(y_i\mid x_i)
\quad = \quad
-\frac{1}{M}\sum_{x,y} N(x,y)\log P(y\mid x)
$$

where $M$ is the total number of examples.


In [ ]:
def build_dataset(words, block_size):
    '''Build context -> next-character examples.

    X shape: (num_examples, block_size)
    Y shape: (num_examples,)
    '''
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


def count_ngrams(X, Y, block_size):
    '''Count table.

    bigram:  shape (27, 27)       = C[prev, next]
    trigram: shape (27, 27, 27)   = C[prevprev, prev, next]
    '''
    N = torch.zeros((vocab_size,) * (block_size + 1), dtype=torch.int64)
    for x, y in zip(X, Y):
        ix = tuple(x.tolist()) + (y.item(),)
        N[ix] += 1
    return N


def probs_from_counts(N, smoothing=1.0):
    '''Convert count table into a conditional probability table.'''
    P = N.float() + smoothing
    P /= P.sum(dim=-1, keepdim=True)
    return P


def _context_tuple(X):
    return tuple(X[:, i] for i in range(X.shape[1]))


def nll_from_examples(P, X, Y):
    '''Average NLL over explicit examples.

    Equivalent to: -mean_i log P[y_i | x_i]
    '''
    probs = P[_context_tuple(X) + (Y,)].clamp_min(1e-30)
    return -probs.log().mean().item()


def nll_from_counts(N, P):
    '''Average NLL using count aggregation.

    Equivalent to: -1/M sum_{x,y} N(x,y) log P(y|x)
    '''
    Nf = N.float()
    return -((Nf * P.clamp_min(1e-30).log()).sum() / Nf.sum()).item()


def perplexity(nll):
    '''exp(NLL), roughly the effective number of choices.'''
    return math.exp(nll)


def sample_count(P, block_size, num=10, max_len=20, generator=None, temperature=1.0):
    '''Sample names from a count-based probability table P.'''
    if generator is None:
        generator = torch.Generator().manual_seed(2147483647)

    out_words = []
    for _ in range(num):
        out = []
        context = [0] * block_size
        while True:
            p = P[tuple(context)]
            if temperature != 1.0:
                logits = p.clamp_min(1e-30).log() / temperature
                p = F.softmax(logits, dim=0)
            ix = torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()
            context = context[1:] + [ix]
            if ix == 0 or len(out) >= max_len:
                break
            out.append(itos[ix])
        out_words.append(''.join(out))
    return out_words


def print_rows(rows, headers):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    print('  '.join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print('  '.join('-' * w for w in widths))
    for row in rows:
        print('  '.join(str(x).ljust(w) for x, w in zip(row, widths)))


#### 3. NLL micro-lab: do not be too surprised by real samples

Start with one concrete sample.

Suppose we have a bigram model $P$. The current character is `e`, and the true next character is `m`. The model assigns a distribution over every possible next character:

$$
P(a\mid e), P(b\mid e), \dots, P(m\mid e), \dots, P(.\mid e)
$$

The NLL for this sample uses only the column for the true character `m`:

$$
-\log P(m\mid e)
$$

Intuition:

- If the model gives `m` high probability, the loss is small.
- If the model gives `m` low probability, the loss is large.

That is the central idea: the less the model believes the event that actually happened, the harder it is penalized.


In [ ]:
# Build full bigram dataset and a smoothed count bigram model.
Xb_all, Yb_all = build_dataset(words, block_size=1)
Nb_all = count_ngrams(Xb_all, Yb_all, block_size=1)
Pb_all = probs_from_counts(Nb_all, smoothing=1.0)

x_char = 'e'
y_char = 'm'
ix = stoi[x_char]
iy = stoi[y_char]
p_correct = Pb_all[ix, iy]
sample_nll = -p_correct.log()

print(f"sample: x='{x_char}' -> y='{y_char}'")
print(f"P({y_char}|{x_char}) = {p_correct.item():.6f}")
print(f"NLL = -log P({y_char}|{x_char}) = {sample_nll.item():.6f}")
print(f"If probability were 0.9,  NLL = {-math.log(0.9):.6f}")
print(f"If probability were 0.001, NLL = {-math.log(0.001):.6f}")


#### 4. Cross-entropy blue-box lab: fixed $x$, sum over possible $y$

After fixing $x=e$, the next character is still not deterministic.

In the real data, we may see:

```text
e -> m
e -> l
e -> r
e -> n
e -> .
...
```

So the blue-box cross-entropy expression:

$$
H(p,p_\theta) = -\sum_y p(y\mid x)\log p_\theta(y\mid x)
$$

means:

> **Fix one context $x$, then take an expectation over all possible next characters $y$.**

With finite data, the true $p(y\mid e)$ is unknown, so we approximate it with counts:

$$
\hat p(y\mid e) = \frac{N(e,y)}{N(e)}
$$

Then these two computations match:

- Compute NLL for every sample whose context is `e`, then average.
- Count all `e -> y` transitions first, then compute a count-weighted cross-entropy.


In [ ]:
def context_cross_entropy_demo_bigram(context_char='e', smoothing=1.0):
    ix = stoi[context_char]
    P_model = probs_from_counts(Nb_all, smoothing=smoothing)

    row_counts = Nb_all[ix].float()
    row_total = row_counts.sum()
    p_emp = row_counts / row_total

    # Cross entropy for this fixed context x=e:
    ce_by_distribution = -(p_emp * P_model[ix].clamp_min(1e-30).log()).sum()

    # Same quantity by explicitly selecting all training examples whose x=e:
    mask = Xb_all[:, 0] == ix
    ce_by_examples = -P_model[ix, Yb_all[mask]].clamp_min(1e-30).log().mean()

    print(f"context x = '{context_char}'")
    print(f"number of examples with x='{context_char}': {int(mask.sum())}")
    print(f"CE by distribution over y: {ce_by_distribution.item():.6f}")
    print(f"Average NLL over examples: {ce_by_examples.item():.6f}")
    print(f"Are they equal? {torch.allclose(ce_by_distribution, ce_by_examples)}")

    vals, inds = torch.topk(p_emp, 8)
    print('\nTop empirical next chars after', repr(context_char))
    for v, j in zip(vals, inds):
        print(f"  {itos[j.item()]!r}: {v.item():.4f}")

context_cross_entropy_demo_bigram('e', smoothing=1.0)


#### 5. Dataset-level equality: sample average = count-weighted average

Example-wise form:

$$
L(\theta)= -\frac{1}{M}\sum_i \log p_\theta(y_i\mid x_i)
$$

Count-aggregated form:

$$
L(\theta)= -\frac{1}{M}\sum_{x,y} N(x,y)\log p_\theta(y\mid x)
$$

For discrete bigram/trigram models, these are exactly equivalent.

This connects:

- explicit bigram log probabilities in Karpathy's notebook.
- count-weighted loss from a statistical table.
- cross-entropy as an expectation over possible next characters.


In [ ]:
loss_examples = nll_from_examples(Pb_all, Xb_all, Yb_all)
loss_counts = nll_from_counts(Nb_all, Pb_all)

print(f'bigram average NLL by examples: {loss_examples:.12f}')
print(f'bigram average NLL by counts:   {loss_counts:.12f}')
print(f'perplexity = exp(NLL):          {perplexity(loss_examples):.4f}')


### E01 - Train a trigram language model

Original prompt:

> Train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; did it improve over a bigram model?

#### What are we trying to understand?

The difference between a bigram and a trigram is not just a different tensor shape:

```text
bigram:  x = previous 1 character
trigram: x = previous 2 characters
```

A trigram model sees one extra character, so it should usually be less surprised by the next character, which means lower loss.

For example:

```text
bigram sees:  m -> ?
trigram sees: em -> ?
```

The context `em -> m` is more specific than `m -> ?`.


In [ ]:
Xb, Yb = build_dataset(words, block_size=1)
Xt, Yt = build_dataset(words, block_size=2)

print('bigram dataset:', Xb.shape, Yb.shape)
print('trigram dataset:', Xt.shape, Yt.shape)

print('\nFirst 8 trigram examples:')
for x, y in zip(Xt[:8], Yt[:8]):
    ctx = ''.join(itos[i.item()] for i in x)
    print(f'{ctx} -> {itos[y.item()]}')


In [ ]:
Nb = count_ngrams(Xb, Yb, block_size=1)
Nt = count_ngrams(Xt, Yt, block_size=2)

Pb = probs_from_counts(Nb, smoothing=1.0)
Pt = probs_from_counts(Nt, smoothing=1.0)

bigram_loss = nll_from_examples(Pb, Xb, Yb)
trigram_loss = nll_from_examples(Pt, Xt, Yt)

rows = [
    ('bigram count model', f'{bigram_loss:.4f}', f'{perplexity(bigram_loss):.2f}'),
    ('trigram count model', f'{trigram_loss:.4f}', f'{perplexity(trigram_loss):.2f}'),
]
print_rows(rows, headers=('model', 'loss/NLL', 'perplexity'))


In [ ]:
g_sample = torch.Generator().manual_seed(2147483647)
print('bigram samples')
for w in sample_count(Pb, block_size=1, num=10, generator=g_sample):
    print(w)

print('\ntrigram samples')
g_sample = torch.Generator().manual_seed(2147483647)
for w in sample_count(Pt, block_size=2, num=10, generator=g_sample):
    print(w)


#### E01 insight

If the trigram loss is lower than the bigram loss, it is not because the word "trigram" is magically smarter. It is because:

> **The input $x$ contains more information about $y$.**

The remaining bigram NLL tells us that, after seeing only 1 previous character, there is still a lot of uncertainty about the next character.

The lower trigram NLL tells us that, after seeing one more character of context, the model is less surprised.


### E02 - Train / Dev / Test split

Original prompt:

> Split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

#### The roles of train, dev, and test

Exam analogy:

- **train set**: textbook and homework. The model is allowed to see it and learn from it.
- **dev set**: practice exam. We use it to make choices, such as bigram vs trigram, smoothing strength, or training steps.
- **test set**: final exam. We look at it once after all choices are made, to estimate performance on genuinely new data.

The key rule:

> **Do not tune on the test set.**

Once you repeatedly inspect test performance and change smoothing or model choices based on it, the test set has become another dev set. It no longer represents unseen data.


In [ ]:
g_split = torch.Generator().manual_seed(2147483647)
perm = torch.randperm(len(words), generator=g_split).tolist()

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = [words[i] for i in perm[:n1]]
dev_words   = [words[i] for i in perm[n1:n2]]
test_words  = [words[i] for i in perm[n2:]]

print('num train/dev/test words:', len(train_words), len(dev_words), len(test_words))
print('first few train words:', train_words[:8])


In [ ]:
# Build bigram splits
Xbtr, Ybtr = build_dataset(train_words, block_size=1)
Xbd,  Ybd  = build_dataset(dev_words,   block_size=1)
Xbte, Ybte = build_dataset(test_words,  block_size=1)

# Build trigram splits
Xttr, Yttr = build_dataset(train_words, block_size=2)
Xtd,  Ytd  = build_dataset(dev_words,   block_size=2)
Xtte, Ytte = build_dataset(test_words,  block_size=2)

# Fit count models on train only
Nb_train = count_ngrams(Xbtr, Ybtr, block_size=1)
Nt_train = count_ngrams(Xttr, Yttr, block_size=2)

Pb_train = probs_from_counts(Nb_train, smoothing=1.0)
Pt_train = probs_from_counts(Nt_train, smoothing=1.0)

rows = [
    ('bigram',
     f'{nll_from_examples(Pb_train, Xbtr, Ybtr):.4f}',
     f'{nll_from_examples(Pb_train, Xbd,  Ybd):.4f}',
     f'{nll_from_examples(Pb_train, Xbte, Ybte):.4f}'),
    ('trigram',
     f'{nll_from_examples(Pt_train, Xttr, Yttr):.4f}',
     f'{nll_from_examples(Pt_train, Xtd,  Ytd):.4f}',
     f'{nll_from_examples(Pt_train, Xtte, Ytte):.4f}'),
]
print_rows(rows, headers=('model', 'train', 'dev', 'test'))


#### E02 insight

There are two things to check.

##### 1. Does the trigram also beat the bigram on dev/test?

If yes, the trigram is not merely memorizing the training set. It is using the longer context to learn patterns that transfer.

##### 2. Is there a gap between train and dev/test?

If:

```text
train loss < dev loss
```

then the model is more familiar with the training set and may be starting to overfit.

The bigram has few parameters, so train/dev/test are usually close. The trigram has more parameters and many sparse contexts, so a gap is more likely.


### E03 - Tune smoothing using dev set

Original prompt:

> Use the dev set to tune the strength of smoothing for the trigram model. Try many possibilities and see which one works best based on dev set loss. Take the best setting and evaluate on the test set once at the end.

#### What is smoothing?

If the trigram context `xz` appears only once in the training set, and the next character is `a`, a model with no smoothing becomes overconfident:

```text
P(a | xz) = 1
P(other | xz) = 0
```

That is excellent for train loss, but dangerous for future data.

Smoothing gives every character a small baseline probability:

> Trust the data, but do not blindly trust it.

#### What does dev do here?

The dev set is used to:

> **choose the smoothing strength.**

The test set is not inspected until the end.


In [ ]:
smoothings = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
results = []

for s in smoothings:
    P = probs_from_counts(Nt_train, smoothing=s)
    train_loss = nll_from_examples(P, Xttr, Yttr)
    dev_loss = nll_from_examples(P, Xtd, Ytd)
    results.append((s, train_loss, dev_loss))

rows = [(s, f'{tr:.4f}', f'{dv:.4f}') for s, tr, dv in results]
print_rows(rows, headers=('smoothing', 'train', 'dev'))

best_s, best_train_loss, best_dev_loss = min(results, key=lambda x: x[2])
print('\nbest smoothing by dev:', best_s)
print('best dev loss:', best_dev_loss)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([r[0] for r in results], [r[1] for r in results], marker='o', label='train')
plt.plot([r[0] for r in results], [r[2] for r in results], marker='o', label='dev')
plt.xscale('log')
plt.xlabel('trigram smoothing')
plt.ylabel('NLL loss')
plt.title('Smoothing: trust train data, but not too much')
plt.legend()
plt.grid(True, alpha=0.3)


In [ ]:
Pbest = probs_from_counts(Nt_train, smoothing=best_s)
best_test_loss = nll_from_examples(Pbest, Xtte, Ytte)

print(f'best smoothing: {best_s}')
print(f'train loss: {nll_from_examples(Pbest, Xttr, Yttr):.4f}')
print(f'dev loss:   {nll_from_examples(Pbest, Xtd,  Ytd):.4f}')
print(f'test loss:  {best_test_loss:.4f}')


#### E03 insight

Usually you will see this pattern:

- Too little smoothing: train loss is very low, but dev may not be best because the model trusts training accidents too much.
- Too much smoothing: train and dev both get worse because the model becomes too conservative and too uniform.
- Some middle value: dev loss is best.

This is the smallest version of hyperparameter tuning / regularization.

One-sentence rule:

> **The dev set is for making choices; the test set is for final reporting.**


### E04 - Replace `F.one_hot` with row indexing

Original prompt:

> We saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

#### What is E04 really doing?

The original neural bigram implementation was:

```python
xenc = F.one_hot(xs, num_classes=27).float()
logits = xenc @ W
```

If the current character is `e`, the one-hot vector is:

```text
[0,0,0,0,0,1,0,...,0]
```

Multiplying it by $W$ simply selects the row of $W$ for `e`:

```python
logits = W[xs]
```

So E04 does not change the model. It changes the implementation:

```text
teaching version: one-hot @ W
practical version: W[index]
```

This step is the precursor to understanding embedding lookup.


In [ ]:
xs = Xbtr[:, 0]
ys = Ybtr
num = xs.nelement()

g_w = torch.Generator().manual_seed(2147483647)
W = torch.randn((vocab_size, vocab_size), generator=g_w, requires_grad=True)

xsmall = xs[:8]
xenc = F.one_hot(xsmall, num_classes=vocab_size).float()
logits_one_hot = xenc @ W
logits_indexed = W[xsmall]

print('one_hot @ W equals W[index]?', torch.allclose(logits_one_hot, logits_indexed))
print('shapes:', logits_one_hot.shape, logits_indexed.shape)


In [ ]:
# Train neural bigram using row indexing.
# This is mathematically the same model class as one-hot @ W, just cheaper.

g_w = torch.Generator().manual_seed(2147483647)
W_indexed = torch.randn((vocab_size, vocab_size), generator=g_w, requires_grad=True)

for k in range(100):
    logits = W_indexed[xs]
    probs = F.softmax(logits, dim=1)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W_indexed**2).mean()

    W_indexed.grad = None
    loss.backward()
    W_indexed.data += -50 * W_indexed.grad

    if k % 20 == 0 or k == 99:
        print(k, loss.item())


In [ ]:
@torch.no_grad()
def eval_neural_bigram_logits(W, X, Y):
    logits = W[X[:, 0]]
    return F.cross_entropy(logits, Y).item()

print(f'train loss: {eval_neural_bigram_logits(W_indexed, Xbtr, Ybtr):.4f}')
print(f'dev loss:   {eval_neural_bigram_logits(W_indexed, Xbd,  Ybd):.4f}')
print(f'test loss:  {eval_neural_bigram_logits(W_indexed, Xbte, Ybte):.4f}')


#### E04 insight

`F.one_hot(xs) @ W` and `W[xs]` express the same mathematical operation.

The difference is implementation cost:

- one-hot explicitly creates many zeros.
- indexing directly looks up rows, uses less memory, and is closer to real deep learning code.

Later, when you see an embedding lookup like:

```python
C[X]
```

you should remember E04:

> **A discrete token enters a neural network by selecting a row from a parameter table.**


### E05 - Replace manual NLL with `F.cross_entropy`

Original prompt:

> Look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

#### Manual NLL

```python
probs = softmax(logits)
loss = -log(probs[range(num), targets]).mean()
```

#### PyTorch version

```python
loss = F.cross_entropy(logits, targets)
```

They are mathematically the same object: softmax + NLL.

Important: `F.cross_entropy` expects raw logits. Do not apply softmax first.


In [ ]:
with torch.no_grad():
    logits = W_indexed[xs[:1024]]
    targets = ys[:1024]

    probs = F.softmax(logits, dim=1)
    manual_loss = -probs[torch.arange(targets.nelement()), targets].log().mean()
    ce_loss = F.cross_entropy(logits, targets)

print('manual softmax + NLL:', manual_loss.item())
print('F.cross_entropy:     ', ce_loss.item())
print('close?', torch.allclose(manual_loss, ce_loss, atol=1e-6))


In [ ]:
# Train the same neural bigram using F.cross_entropy.
g_w = torch.Generator().manual_seed(2147483647)
W_ce = torch.randn((vocab_size, vocab_size), generator=g_w, requires_grad=True)

for k in range(100):
    logits = W_ce[xs]
    loss = F.cross_entropy(logits, ys) + 0.01 * (W_ce**2).mean()

    W_ce.grad = None
    loss.backward()
    W_ce.data += -50 * W_ce.grad

    if k % 20 == 0 or k == 99:
        print(k, loss.item())


In [ ]:
rows = [
    ('indexed + manual NLL',
     f'{eval_neural_bigram_logits(W_indexed, Xbtr, Ybtr):.4f}',
     f'{eval_neural_bigram_logits(W_indexed, Xbd,  Ybd):.4f}',
     f'{eval_neural_bigram_logits(W_indexed, Xbte, Ybte):.4f}'),
    ('indexed + cross_entropy',
     f'{eval_neural_bigram_logits(W_ce, Xbtr, Ybtr):.4f}',
     f'{eval_neural_bigram_logits(W_ce, Xbd,  Ybd):.4f}',
     f'{eval_neural_bigram_logits(W_ce, Xbte, Ybte):.4f}'),
]
print_rows(rows, headers=('model', 'train', 'dev', 'test'))


#### E05 insight

`F.cross_entropy` is preferred because:

1. It is numerically stable. It does not naively call `exp(logits)` and overflow.
2. It is shorter and harder to write incorrectly.
3. It is the standard PyTorch API.
4. Internally, it fuses `log_softmax + negative log likelihood`.

For learning, you should understand the hand-written softmax + NLL. For engineering, use `F.cross_entropy`.


### Final synthesis: one mental model for the whole notebook

The whole notebook can be compressed into one thread:

1. Split names into supervised examples: $x_i = \text{context}$ and $y_i = \text{next character}$.
2. The model outputs the next-character distribution: $p_\theta(y\mid x_i)$.
3. NLL measures surprise at the real target: $-\log p_\theta(y_i\mid x_i)$.
4. A count table estimates the empirical conditional distribution: $\hat p(y\mid x)=N(x,y)/N(x)$.
5. A neural model learns $p_\theta(y\mid x)$ through optimization.
6. Cross-entropy measures average surprise from a true/teacher distribution to the model distribution: $H(p,p_\theta)=-\sum_y p(y\mid x)\log p_\theta(y\mid x)$.
7. Train/dev/test separate fitting from generalization: $\mathcal{D}_{train}$, $\mathcal{D}_{dev}$, and $\mathcal{D}_{test}$.
8. Smoothing/regularization prevents over-trusting the training set: $N(x,y)\mapsto N(x,y)+s$ or $L(\theta)\mapsto L(\theta)+\lambda R(\theta)$.
9. One-hot matrix multiplication is row lookup: $\mathrm{one\_hot}(x)W = W_x$.
10. Temperature controls sampling behavior: $P_T(y\mid x)=\mathrm{softmax}(\log P(y\mid x)/T)$.

The most important sentence:

> **A loss function is not magic. It asks: under my probabilistic modeling assumption, when the real data appears, how surprised is my model?**

For a character language model:

$$
y_i\mid x_i \sim \mathrm{Categorical}(p_\theta(y\mid x_i))
$$

therefore:

$$
L(\theta)= -\frac{1}{M}\sum_i \log p_\theta(y_i\mid x_i)
$$

This is NLL, which is also cross-entropy for one-hot classification targets.
